# SSHR - MICCAI 2026

## Hướng dẫn chạy
| Lần | Các cell cần chạy |
|-----|-------------------|
| **Lần đầu** | Cell 1 → Cell 2 → (restart nếu được yêu cầu) → Cell 1 → Cell 3 → Cell 4 |
| **Lần sau** | Cell 1 → Cell 3 → Cell 4 |


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Mount Drive + Clone/check repo + CD vào repo
# Chạy lại cell này sau mỗi lần restart
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_PATH = '/content/drive/MyDrive/SSHR'

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/trongduc-nguyen/SSHR.git "{REPO_PATH}"
    print("✅ Cloned xong")
else:
    print("✅ Repo đã có sẵn, bỏ qua clone")

os.chdir(REPO_PATH)
print("📂 Working dir:", os.getcwd())

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 2 — Cài packages (CHỈ chạy 1 lần đầu)
# Nếu Colab yêu cầu restart → bấm Restart → chạy lại Cell 1 → BỎ QUA cell này
# ══════════════════════════════════════════════════════════════
!pip install -r requirements.txt
!pip install mxnet==1.9.1
# numpy==1.23.5 không build được trên Python 3.13 → dùng 1.26.4
!pip install "numpy>=1.24,<2.0"

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 3 — Kiểm tra GPU
# ══════════════════════════════════════════════════════════════
import torch
gpu_name = torch.cuda.get_device_name(0)
bf16_ok  = torch.cuda.is_bf16_supported()
print(f"GPU  : {gpu_name}")
print(f"bf16 : {bf16_ok}")
print(f"→ Dùng amp-dtype: {'bf16' if bf16_ok else 'fp16'}")

In [ ]:
%%bash
# ══════════════════════════════════════════════════════════════
# Cell 4 — Train LUAD
# Chỉnh DATASET_DIR khớp tên folder thực tế trên Drive
# ══════════════════════════════════════════════════════════════
DATA_ROOT=/content/drive/MyDrive/datasets
DATASET=luad
DATASET_DIR=LUAD-HistoSeg
SEED=11
AMP=fp16   # đổi sang bf16 nếu Cell 3 báo bf16: True

cd /content/drive/MyDrive/SSHR

python train_sshr.py \
  --dataset        "${DATASET}" \
  --seed           "${SEED}" \
  --max_epoches    25 \
  --weights        init_weights/ilsvrc-cls_rna-a1_cls1000_ep-0001.params \
  --trainroot      "${DATA_ROOT}/${DATASET_DIR}/training" \
  --testroot       "${DATA_ROOT}/${DATASET_DIR}/test" \
  --valroot        "${DATA_ROOT}/${DATASET_DIR}/val" \
  --save_folder    "checkpoints_${DATASET}_seed${SEED}" \
  --eval_every     0 \
  --save-last-k-checkpoints 0 \
  --n_class        4 \
  --img_size       224 \
  --batch_size     32 \
  --num_workers    2 \
  --amp-dtype      "${AMP}"

In [ ]:
%%bash
# ══════════════════════════════════════════════════════════════
# Cell 5 — Train BCSS
# Chỉnh DATASET_DIR khớp tên folder BCSS thực tế trên Drive
# ══════════════════════════════════════════════════════════════
DATA_ROOT=/content/drive/MyDrive/datasets
DATASET=bcss
DATASET_DIR=BCSS-WSSS
SEED=11
AMP=fp16   # đổi sang bf16 nếu Cell 3 báo bf16: True

cd /content/drive/MyDrive/SSHR

python train_sshr.py \
  --dataset        "${DATASET}" \
  --seed           "${SEED}" \
  --max_epoches    25 \
  --weights        init_weights/ilsvrc-cls_rna-a1_cls1000_ep-0001.params \
  --trainroot      "${DATA_ROOT}/${DATASET_DIR}/training" \
  --testroot       "${DATA_ROOT}/${DATASET_DIR}/test" \
  --valroot        "${DATA_ROOT}/${DATASET_DIR}/val" \
  --save_folder    "checkpoints_${DATASET}_seed${SEED}" \
  --eval_every     0 \
  --save-last-k-checkpoints 0 \
  --n_class        4 \
  --img_size       224 \
  --batch_size     32 \
  --num_workers    2 \
  --amp-dtype      "${AMP}"